# Three-model spoiler comparison
GPU inference only: owned MiniLM review-only epoch2, pinned Zritze RoBERTa, Qwen2.5-3B-Instruct without RAG. Text-only inputs, 147 development excerpts. Human/AI labels reported separately; incomplete human metadata and unverified grouping mean exploratory agreement only. Held-out test is excluded. No threshold fitting. Upload the owned inputs ZIP when asked. Files under /content are ephemeral: download caches after each stage. No API keys, Drive mounts, training, or paid provisioning.


In [ ]:
import subprocess,sys
subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers==4.57.1','safetensors>=0.4.3'])
import torch
assert torch.cuda.is_available(), 'Select a T4 GPU runtime'
print(torch.cuda.get_device_name(0))


In [ ]:
from google.colab import files
from pathlib import Path
import hashlib,io,zipfile
name='three-model-comparison-inputs.zip'
package=Path('/content')/name
if not package.exists():
    uploaded=files.upload()
    assert name in uploaded, 'Upload '+name
payload=package.read_bytes()
assert hashlib.sha256(payload).hexdigest()=='179aed30de3c97fc4e4f8a339471bfc74fdf871f6fe36df4035e4d76a635bb6a'
ROOT=Path('/content/three-model-comparison');ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    assert all((ROOT/r.filename).resolve().is_relative_to(ROOT.resolve()) for r in z.infolist())
    z.extractall(ROOT)
OUT=ROOT/'results';OUT.mkdir(exist_ok=True)


In [ ]:
import shutil
def run_model(name):
    process=subprocess.Popen([sys.executable,str(ROOT/'compare_three_models.py'),'predict','--data',str(ROOT/'dev-blind.jsonl'),'--out',str(OUT),'--model',name,'--minilm',str(ROOT/'minilm-epoch2')],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    for line in process.stdout: print(line,end='',flush=True)
    assert process.wait()==0, 'Model inference failed; inspect output'
def archive():
    shutil.make_archive('/content/three-model-results','zip',OUT)
    files.download('/content/three-model-results.zip')


Run minilm (no training). The LLM download is approximately 6 GB; it runs on the T4, with one model loaded at a time.

In [ ]:
run_model('minilm')
archive()


Run roberta (no training). The LLM download is approximately 6 GB; it runs on the T4, with one model loaded at a time.

In [ ]:
run_model('roberta')
archive()


Run llm (no training). The LLM download is approximately 6 GB; it runs on the T4, with one model loaded at a time.

In [ ]:
run_model('llm')
archive()


In [ ]:
completed=subprocess.run([sys.executable,str(ROOT/'compare_three_models.py'),'report','--data',str(ROOT/'dev-blind.jsonl'),'--annotations',str(ROOT/'annotations.json'),'--out',str(OUT)],capture_output=True,text=True)
print(completed.stdout)
if completed.returncode: print(completed.stderr)
completed.check_returncode()
archive()
